# Quickstart

`osmfinder` finds OpenStreetMap extracts (`*.osm.pbf` files) that other people cut out of the planet and publish: by their name, by a point, or by a geometry they have to cover.

In [ ]:
%pip install osmfinder basemaplibre folium geopandas mapclassify matplotlib

In [ ]:
import basemaplibre as bml
import folium
import matplotlib.pyplot as plt
from shapely.geometry import box

import osmfinder

## Find by name

Names repeat between the sources. The smallest of the matching extracts is selected, with a warning about the others:

In [ ]:
osmfinder.find("Liechtenstein")

Narrow the search down to one source, or to a few of them, to get the one you want:

In [ ]:
result = osmfinder.find("Liechtenstein", source="Geofabrik")
result.extract

`matched_extracts` has all the matches. `select_first_match=False` would raise an error instead of choosing one of them.

In [ ]:
matches = osmfinder.find("Liechtenstein").matched_extracts
osmfinder.extracts_to_geodataframe(matches).drop(columns="geometry")

## Find by a point

Every extract that covers the point, from the smallest to the largest. Here is Prague:

In [ ]:
point = (14.4378, 50.0755)
extracts = osmfinder.find_extracts_covering_point(point, source=["Geofabrik", "BBBike"])
[extract.id for extract in extracts]

In [ ]:
gdf = osmfinder.extracts_to_geodataframe(extracts)
m = gdf[:-1].explore(
    column="id",
    cmap="tab10",
    tiles=folium.TileLayer("OpenStreetMap", opacity=0.3),
    style_kwds={"fillOpacity": 0.1},
)
folium.Marker(point[::-1]).add_to(m)
m

## Find by a geometry

Any Shapely geometry works, in longitude and latitude. This box lies where Austria, Germany, Liechtenstein and Switzerland meet:

In [ ]:
geometry = box(9.4, 47.2, 9.8, 47.6)
result = osmfinder.find(geometry)
result

No extract of a country had to be taken: smaller ones cover the box together. A map shows how:

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9), dpi=120)
result.plot(ax=ax)
bml.add_basemap(ax, crs=4326, source="OpenFreeMap.Positron", labels="above")
plt.show()

[Covering a geometry](covering.ipynb) explains how the extracts are picked and how to change it.

## Download

`download` takes the same arguments as `find`, or the result of one:

In [ ]:
download = osmfinder.download("Monaco", source="Geofabrik", progressbar=False)
download

In [ ]:
for path in download.download_paths:
    print(path, f"{path.stat().st_size / 1e6:.2f} MB")

Tools like [QuackOSM](https://github.com/kraina-ai/quackosm), [osmium](https://osmcode.org/osmium-tool/) or [GDAL](https://gdal.org/) read those files.